# 05 Clean: find the artifacts, then write rules
Every cleaning rule must be motivated by an artifact you actually observed, with an example (accession number and SOI row), and every rule gets a test.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
sub, tag = load.load_sub(RDIR), load.load_tag(RDIR)
header = load.soi_header(RDIR)
colmap = mapping.map_soi_columns(header, tag)
soi = load.load_soi(RDIR, usecols=mapping.soi_usecols(header, colmap))   # only the columns we use
soi = soi[soi["adsh"].isin(sub["adsh"])]                                  # drop superseded filings
canon = holdings.split_periods(mapping.to_canonical(soi, colmap), sub)  # current period from SUB
cur_rows = canon[canon["is_current"]]
no_id = cur_rows[cur_rows["identifier"].isna() & cur_rows["fair_value"].notna()]
print("Current rows with a fair value but no identifier (subtotals, or holdings on a custom axis):")
print(no_id[["name", "investment_type", "industry", "affiliation", "fair_value"]].to_string(index=False))

In [ ]:
h = holdings.coalesce_holdings(canon)
cur = h[h["is_current"]]
for rule in (clean.rule_totals, clean.rule_cash):
    drop, why = rule(cur)
    print(f"{rule.__name__}: {why} -> {int(drop.sum())} row(s)")
    print(cur[drop][["name", "identifier", "fair_value"]].to_string(index=False))

**Caution on cash.** Whether a filer's reported investment total includes cash equivalents and money-market funds varies. The fixture's BRAVO excludes them, so `rule_cash` is right for BRAVO; on your release, check each filer's balance sheet before you drop such lines.

In [ ]:
print("Filings whose SOI uses a custom tag or axis (the SEC codes cstm as 1 = yes, 2 = no):")
print(canon.groupby("name")["cstm"].first().map({"1": "yes", "2": "no"}).to_string())
print("\nHoldings with no fair value in the SOI:")
print(cur[cur["fair_value"].isna()][["name", "identifier"]].to_string(index=False))

In [ ]:
kept, log = clean.apply_rules(cur, [clean.rule_totals, clean.rule_cash])
print(f"Holdings kept: {len(kept)} of {len(cur)}")
log

In [ ]:
# Measure only AFTER cleaning: a total row left in would sit in the denominator.
debt = kept[clean.is_debt(kept)].groupby("name")["fair_value"].sum()
allfv = kept.groupby("name")["fair_value"].sum()
print("Debt share of fair value by BDC (low values suggest a venture or equity-heavy lender):")
print((debt.reindex(allfv.index).fillna(0) / allfv).round(3).to_string())

**Your turn.** Your release will show artifacts the fixture does not. For each one, add a rule function to your own package (same shape as `rule_totals`: return a drop mask and a one-line artifact description), add it to the rule list, and add a test that fails if the rule is removed. Typical candidates: rows that restate others, negative fair values for unfunded commitments, values the SOI omits because the filer used a custom tag.